In [1]:

import pandas as pd
import numpy as np
import re

invoices = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\invoices.csv")
payments = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\gateway_payments.csv")


Q2_START = pd.Timestamp("2025-04-01", tz="Asia/Kolkata")
Q2_END = pd.Timestamp("2025-06-30 23:59:59.999999", tz="Asia/Kolkata")
Q3_END = pd.Timestamp("2025-09-30 23:59:59.999999", tz="Asia/Kolkata")


def normalize_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["match_key"] = invoices["invoice_id"].apply(normalize_id)
payments["match_key"] = payments["invoice_ref"].apply(normalize_id)


invoices["invoice_date"] = pd.to_datetime(
    invoices["invoice_date"], errors="coerce"
)


if invoices["invoice_date"].dt.tz is None:
    invoices["invoice_date"] = (
        invoices["invoice_date"].dt.tz_localize("Asia/Kolkata")
    )
else:
    invoices["invoice_date"] = (
        invoices["invoice_date"].dt.tz_convert("Asia/Kolkata")
    )


if "created_at_ist" in payments.columns:
    payments["payment_date"] = pd.to_datetime(
        payments["created_at_ist"], errors="coerce"
    )
    if payments["payment_date"].dt.tz is None:
        payments["payment_date"] = (
            payments["payment_date"].dt.tz_localize("Asia/Kolkata")
        )
    else:
        payments["payment_date"] = (
            payments["payment_date"].dt.tz_convert("Asia/Kolkata")
        )

elif "created_at_epoch" in payments.columns:
    payments["payment_date"] = pd.to_datetime(
        pd.to_numeric(payments["created_at_epoch"], errors="coerce"),
        unit="s",
        utc=True
    ).dt.tz_convert("Asia/Kolkata")

else:
    payments["payment_date"] = pd.to_datetime(
        payments["created_at_utc"], errors="coerce", utc=True
    ).dt.tz_convert("Asia/Kolkata")


q2_invoices = invoices.loc[
    invoices["invoice_date"].between(Q2_START, Q2_END)
].copy()

q2_invoices["amount_rs"] = pd.to_numeric(
    q2_invoices["amount_rs"], errors="coerce"
).fillna(0)


payments["amount_rs"] = pd.to_numeric(
    payments["amount_rs"], errors="coerce"
)

captured = payments.loc[
    payments["status"].astype(str).str.strip().str.lower()
    .eq("captured")
].copy()


if "payment_id" in captured.columns:
    captured = captured.drop_duplicates(
        subset=["payment_id"], keep="first"
    )

q2_keys = set(q2_invoices["match_key"].dropna())
linked = captured.loc[
    captured["match_key"].isin(q2_keys)
    & captured["payment_date"].notna()
].copy()


paid_by_q2 = (
    linked.loc[linked["payment_date"] <= Q2_END]
    .groupby("match_key")["amount_rs"]
    .sum()
    .rename("paid_by_q2_rs")
)


paid_in_q3 = (
    linked.loc[
        (linked["payment_date"] > Q2_END)
        & (linked["payment_date"] <= Q3_END)
    ]
    .groupby("match_key")["amount_rs"]
    .sum()
    .rename("paid_in_q3_rs")
)


timing = q2_invoices.copy()

timing = timing.join(paid_by_q2, on="match_key")
timing = timing.join(paid_in_q3, on="match_key")

timing["paid_by_q2_rs"] = timing["paid_by_q2_rs"].fillna(0)
timing["paid_in_q3_rs"] = timing["paid_in_q3_rs"].fillna(0)


timing["outstanding_at_q2_end_rs"] = (
    timing["amount_rs"] - timing["paid_by_q2_rs"]
).clip(lower=0)


timing["later_payment_applied_rs"] = np.minimum(
    timing["outstanding_at_q2_end_rs"],
    timing["paid_in_q3_rs"].clip(lower=0)
)


timing["still_outstanding_after_q3_rs"] = (
    timing["outstanding_at_q2_end_rs"]
    - timing["later_payment_applied_rs"]
)

timing["timing_status"] = np.select(
    [
        timing["outstanding_at_q2_end_rs"].eq(0),
        timing["still_outstanding_after_q3_rs"].eq(0)
    ],
    [
        "Paid by Q2 end",
        "Paid after Q2 during Q3"
    ],
    default="Still outstanding after Q3 check"
)

summary = pd.Series({
    "Q2 invoices (count)": len(timing),
    "Q2 invoiced amount (Rs)": timing["amount_rs"].sum(),
    "Outstanding at Q2 end (Rs)": (
        timing["outstanding_at_q2_end_rs"].sum()
    ),
    "Outstanding invoices at Q2 end (count)": (
        timing["outstanding_at_q2_end_rs"].gt(0).sum()
    ),
    "Later payments applied (Rs)": (
        timing["later_payment_applied_rs"].sum()
    ),
    "Invoices fully recovered during Q3 (count)": (
        (
            timing["outstanding_at_q2_end_rs"].gt(0)
            & timing["still_outstanding_after_q3_rs"].eq(0)
        ).sum()
    ),
    "Still outstanding after Q3 (Rs)": (
        timing["still_outstanding_after_q3_rs"].sum()
    ),
    "Still outstanding after Q3 (count)": (
        timing["still_outstanding_after_q3_rs"].gt(0).sum()
    )
})

print("\nTIMING ANALYSIS SUMMARY")
print(summary.to_string())


unpaid_at_q2_end = timing.loc[
    timing["outstanding_at_q2_end_rs"] > 0,
    [
        "invoice_id",
        "invoice_date",
        "amount_rs",
        "paid_by_q2_rs",
        "outstanding_at_q2_end_rs",
        "paid_in_q3_rs",
        "later_payment_applied_rs",
        "still_outstanding_after_q3_rs",
        "timing_status"
    ]
].sort_values("still_outstanding_after_q3_rs", ascending=False)

print("\nINVOICES UNPAID AT Q2 END")
print(unpaid_at_q2_end.head(20).to_string(index=False))



TIMING ANALYSIS SUMMARY
Q2 invoices (count)                               77915.0
Q2 invoiced amount (Rs)                       118105523.0
Outstanding at Q2 end (Rs)                      9088446.0
Outstanding invoices at Q2 end (count)             6065.0
Later payments applied (Rs)                     4407518.0
Invoices fully recovered during Q3 (count)         2990.0
Still outstanding after Q3 (Rs)                 4680928.0
Still outstanding after Q3 (count)                 3075.0

INVOICES UNPAID AT Q2 END
     invoice_id              invoice_date  amount_rs  paid_by_q2_rs  outstanding_at_q2_end_rs  paid_in_q3_rs  later_payment_applied_rs  still_outstanding_after_q3_rs                    timing_status
INV-2025-084920 2025-04-07 00:00:00+05:30     7997.0            0.0                    7997.0            0.0                       0.0                         7997.0 Still outstanding after Q3 check
INV-2025-117463 2025-05-15 00:00:00+05:30     7348.0            0.0                   

In [2]:

payments["status_clean"] = (
    payments["status"].astype(str).str.strip().str.lower()
)


invoice_payment_counts = (
    payments.groupby("match_key")
    .agg(
        payment_attempts=("payment_id", "size"),
        distinct_payment_ids=("payment_id", "nunique"),
        statuses=("status_clean",
                  lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

multiple_attempts = invoice_payment_counts[
    invoice_payment_counts["distinct_payment_ids"] > 1
].copy()

print("Invoices with multiple payment IDs:",
      len(multiple_attempts))

print(multiple_attempts.head(20).to_string(index=False))


payment_status_by_invoice = (
    payments.groupby("match_key")
    .agg(
        has_failed=("status_clean",
                    lambda x: x.eq("failed").any()),
        has_captured=("status_clean",
                      lambda x: x.eq("captured").any()),
        payment_ids=("payment_id",
                     lambda x: x.nunique()),
        total_rows=("payment_id", "size")
    )
    .reset_index()
)

retried_successfully = payment_status_by_invoice[
    payment_status_by_invoice["has_failed"]
    & payment_status_by_invoice["has_captured"]
].copy()

print("\nInvoices with both failed and captured attempts:",
      len(retried_successfully))

print(retried_successfully.head(20).to_string(index=False))


duplicate_payment_rows = payments[
    payments.duplicated("payment_id", keep=False)
].sort_values("payment_id")

print("\nRows sharing a payment ID:",
      len(duplicate_payment_rows))

print(duplicate_payment_rows.head(30).to_string(index=False))


print("\nPayment status summary:")
print(
    payments.groupby("status_clean")
    .agg(
        row_count=("payment_id", "size"),
        distinct_payment_ids=("payment_id", "nunique"),
        amount_rs=("amount_rs", "sum")
    )
    .sort_values("row_count", ascending=False)
    .to_string()
)


Invoices with multiple payment IDs: 19814
    match_key  payment_attempts  distinct_payment_ids         statuses
INV2025000011                 2                     2 captured, failed
INV2025000089                 2                     2 captured, failed
INV2025000129                 2                     2 captured, failed
INV2025000148                 2                     2 captured, failed
INV2025000209                 2                     2 captured, failed
INV2025000248                 2                     2 captured, failed
INV2025000252                 2                     2 captured, failed
INV2025000269                 2                     2 captured, failed
INV2025000287                 2                     2 captured, failed
INV2025000363                 2                     2 captured, failed
INV2025000373                 2                     2 captured, failed
INV2025000398                 2                     2 captured, failed
INV2025000411                 2    

In [3]:

payments["status_clean"] = (
    payments["status"].astype(str).str.strip().str.lower()
)


captured = payments[payments["status_clean"] == "captured"].copy()


captured_by_invoice = (
    captured.groupby("match_key")
    .agg(
        captured_rows=("payment_id", "size"),
        captured_payment_ids=("payment_id", "nunique"),
        captured_amount_rs=("amount_rs", "sum"),
        payment_ids=("payment_id", lambda x: ", ".join(x.astype(str)))
    )
    .reset_index()
)


multiple_captured = captured_by_invoice[
    captured_by_invoice["captured_payment_ids"] > 1
].sort_values("captured_amount_rs", ascending=False)

print("Invoices with multiple captured payment IDs:",
      len(multiple_captured))

print(multiple_captured.head(30).to_string(index=False))

print("\nTotal captured amount across these invoices (Rs):",
      multiple_captured["captured_amount_rs"].sum())


Invoices with multiple captured payment IDs: 1925
    match_key  captured_rows  captured_payment_ids  captured_amount_rs                    payment_ids
INV2026039887              2                     2             12117.0 pay_0000056130, pay_0000056139
INV2025086936              2                     2             10067.0 pay_000001582f, pay_000001584b
INV2026075063              2                     2              9617.0 pay_000005ec4d, pay_000005ec68
INV2025179470              2                     2              9597.0 pay_000002c6b3, pay_000002c6f1
INV2025177652              2                     2              9567.0 pay_000002bf9b, pay_000002bfb7
INV2025216693              2                     2              9467.0 pay_0000035a43, pay_0000035a7a
INV2025300385              2                     2              9267.0 pay_000004a55a, pay_000004a574
INV2026138476              2                     2              9168.0 pay_000006e756, pay_000006e760
INV2025150884              2    

In [5]:

captured_totals = (
    payments[payments["status_clean"] == "captured"]
    .groupby("match_key")
    .agg(
        captured_payment_count=("payment_id", "nunique"),
        total_captured_rs=("amount_rs", "sum"),
        payment_ids=("payment_id", lambda x: ", ".join(x.astype(str)))
    )
    .reset_index()
)

invoice_amounts = (
    invoices.groupby("match_key")
    .agg(invoice_amount_rs=("amount_rs", "first"))
    .reset_index()
)

payment_comparison = invoice_amounts.merge(
    captured_totals,
    on="match_key",
    how="left"
)

payment_comparison["total_captured_rs"] = (
    payment_comparison["total_captured_rs"].fillna(0)
)

payment_comparison["excess_captured_rs"] = (
    payment_comparison["total_captured_rs"]
    - payment_comparison["invoice_amount_rs"]
)

possible_overpayments = payment_comparison[
    payment_comparison["excess_captured_rs"] > 0
].sort_values("excess_captured_rs", ascending=False)

print("Invoices with captured payments above invoice value:",
      len(possible_overpayments))

print("\nTotal possible excess captured (Rs):",
      possible_overpayments["excess_captured_rs"].sum())

print("\nLargest possible overpayments:")
print(
    possible_overpayments[
        [
            "match_key",
            "invoice_amount_rs",
            "total_captured_rs",
            "excess_captured_rs",
            "captured_payment_count",
            "payment_ids"
        ]
    ].head(30).to_string(index=False)
)


Invoices with captured payments above invoice value: 1973

Total possible excess captured (Rs): 3078429.0

Largest possible overpayments:
    match_key  invoice_amount_rs  total_captured_rs  excess_captured_rs  captured_payment_count                    payment_ids
INV2025274710             1120.0             8818.0              7698.0                     2.0 pay_0000044020, pay_000004402b
INV2026112524              330.0             7927.0              7597.0                     2.0 pay_00000680b8, pay_00000680ca
INV2026039887             4549.0            12117.0              7568.0                     2.0 pay_0000056130, pay_0000056139
INV2026138476             1670.0             9168.0              7498.0                     2.0 pay_000006e756, pay_000006e760
INV2026073945             1050.0             8497.0              7447.0                     2.0 pay_000005e7f2, pay_000005e7fb
INV2025038052              600.0             7918.0              7318.0                     2.0 pay_

In [8]:
import pandas as pd
import re

def normalize_id(value):
    if pd.isna(value):
        return None

    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


invoices["invoice_key"] = invoices["invoice_id"].apply(normalize_id)


payments["payment_key"] = payments["invoice_ref_raw"].apply(
    normalize_id
)


valid_invoice_keys = set(
    invoices["invoice_key"].dropna()
)

orphan_payments = payments[
    payments["payment_key"].notna()
    & ~payments["payment_key"].isin(valid_invoice_keys)
].copy()

print("Unmatched payment rows:", len(orphan_payments))

print(
    "Distinct unmatched invoice references:",
    orphan_payments["payment_key"].nunique()
)

orphan_summary = (
    orphan_payments.groupby("payment_key")
    .agg(
        payment_rows=("payment_id", "size"),
        distinct_payment_ids=("payment_id", "nunique"),
        statuses=("status_clean",
                  lambda x: ", ".join(sorted(set(x.dropna())))),
        total_amount_rs=("amount_rs", "sum"),
        first_payment_date=("payment_date", "min"),
        last_payment_date=("payment_date", "max"),
        source_files=("source_file",
                      lambda x: ", ".join(
                          sorted(set(x.dropna().astype(str)))
                      )),
        methods=("method",
                 lambda x: ", ".join(
                     sorted(set(x.dropna().astype(str)))
                 )),
        settlement_count=("settlement_id", "nunique")
    )
    .reset_index()
    .sort_values("payment_rows", ascending=False)
)

print("\nUnmatched reference analysis:")
print(orphan_summary.head(30).to_string(index=False))

Unmatched payment rows: 4765
Distinct unmatched invoice references: 4588

Unmatched reference analysis:
   payment_key  payment_rows  distinct_payment_ids         statuses  total_amount_rs        first_payment_date         last_payment_date                  source_files    methods  settlement_count
 INV2026981315             2                     2 captured, failed           3698.0 2026-03-29 17:14:02+05:30 2026-03-29 17:27:25+05:30 gateway_export_2026-03.csv.gz        upi                 1
INV20251178479             2                     2 captured, failed            700.0 2025-12-01 01:59:51+05:30 2025-12-01 02:09:50+05:30 gateway_export_2025-11.csv.gz        upi                 1
INV20261051287             2                     2 captured, failed            700.0 2026-06-20 17:06:48+05:30 2026-06-20 17:10:39+05:30 gateway_export_2026-06.csv.gz       cash                 1
INV20251132733             2                     2 captured, failed           5500.0 2025-09-30 22:24:30+05:30 2

In [9]:
print(
    orphan_summary.head(30)[
        ["payment_key", "payment_rows", "statuses",
         "total_amount_rs", "source_files"]
    ].to_string(index=False)
)

orphan_summary["reference_length"] = (
    orphan_summary["payment_key"].str.len()
)

print("\nReference length distribution:")
print(orphan_summary["reference_length"].value_counts().sort_index())

orphan_summary["reference_prefix"] = (
    orphan_summary["payment_key"].str[:3]
)

print("\nMost common reference prefixes:")
print(orphan_summary["reference_prefix"].value_counts().head(20))

   payment_key  payment_rows         statuses  total_amount_rs                  source_files
 INV2026981315             2 captured, failed           3698.0 gateway_export_2026-03.csv.gz
INV20251178479             2 captured, failed            700.0 gateway_export_2025-11.csv.gz
INV20261051287             2 captured, failed            700.0 gateway_export_2026-06.csv.gz
INV20251132733             2 captured, failed           5500.0 gateway_export_2025-09.csv.gz
 INV2025996574             2 captured, failed           1760.0 gateway_export_2025-05.csv.gz
INV20251018496             2 captured, failed           3300.0 gateway_export_2025-05.csv.gz
INV20261024648             2 captured, failed           3500.0 gateway_export_2026-05.csv.gz
INV20251183424             2 captured, failed           4198.0 gateway_export_2025-12.csv.gz
 INV2026981233             2 captured, failed           1300.0 gateway_export_2026-04.csv.gz
 INV2025996828             2 captured, failed           2200.0 gateway

In [10]:

valid_keys = invoices["invoice_key"].dropna()
unmatched_keys = orphan_summary["payment_key"].dropna()

print("VALID INVOICE KEY LENGTHS")
print(valid_keys.str.len().value_counts().sort_index())

print("\nUNMATCHED REFERENCE LENGTHS")
print(unmatched_keys.str.len().value_counts().sort_index())


orphan_payments["payment_date"] = pd.to_datetime(
    orphan_payments["payment_date"], errors="coerce"
)

orphan_payments["payment_month"] = (
    orphan_payments["payment_date"].dt.to_period("M")
)

print("\nUNMATCHED PAYMENTS BY MONTH")
print(
    orphan_payments.groupby("payment_month")
    .agg(
        payment_rows=("payment_id", "size"),
        distinct_references=("payment_key", "nunique"),
        captured_rows=(
            "status_clean",
            lambda x: x.eq("captured").sum()
        ),
        failed_rows=(
            "status_clean",
            lambda x: x.eq("failed").sum()
        ),
        amount_field_total=("amount_rs", "sum")
    )
    .to_string()
)


print("\nUNMATCHED PAYMENTS BY SOURCE FILE")
print(
    orphan_payments.groupby("source_file")
    .agg(
        payment_rows=("payment_id", "size"),
        distinct_references=("payment_key", "nunique"),
        captured_rows=(
            "status_clean",
            lambda x: x.eq("captured").sum()
        ),
        failed_rows=(
            "status_clean",
            lambda x: x.eq("failed").sum()
        )
    )
    .sort_values("payment_rows", ascending=False)
    .to_string()
)


print("\nUNMATCHED REFERENCE STATUS COMBINATIONS")
print(
    orphan_summary["statuses"]
    .value_counts(dropna=False)
    .to_string()
)

VALID INVOICE KEY LENGTHS
invoice_key
13    470776
Name: count, dtype: int64

UNMATCHED REFERENCE LENGTHS
payment_key
13    1903
14    2685
Name: count, dtype: int64

UNMATCHED PAYMENTS BY MONTH
               payment_rows  distinct_references  captured_rows  failed_rows  amount_field_total
payment_month                                                                                   
2025-01                 234                  224            224           10            382554.0
2025-02                 270                  256            256           14            429783.0
2025-03                 259                  254            254            5            393764.0
2025-04                 239                  227            227           12            350378.0
2025-05                 270                  260            260           10            413816.0
2025-06                 291                  281            281           10            450436.0
2025-07                 268  

C:\Users\Diksha\AppData\Local\Temp\ipykernel_24968\2532602074.py:18: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  orphan_payments["payment_date"].dt.to_period("M")
